# Explanation Generation Evaluation
We are testing whether the models are able to create the explanation of why system_a and system_b are analogies. 

`Assumption:` We are assuming that the analogy is correct.

`Testing Criteria:`
1.  unfamiliar concept + familiar concept => Explanation , Reasoning
2. unfamiliar concept + familiar concept + unpaired properties => Explanation , Reasoning
2. unfamiliar concept + familiar concept + paired properties => Explanation , Reasoning

In [21]:
import os
from dotenv import load_dotenv
import pandas as pd
load_dotenv()
# Import the easy LLM importer
import sys
import os

# Get the current working directory (since __file__ doesn't exist)
current_dir = os.getcwd()

# Build the path to the mapping_generation folder
mapping_path = os.path.abspath(os.path.join(current_dir, '..', 'mapping_generation'))
sys.path.append(mapping_path)

from easy_llm_importer import create_client, list_available_models, quick_chat, DSPyAdapter
import dspy
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
deepinfra_api_key = os.getenv("DEEPINFRA_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# Create client
client = create_client()

In [22]:
# See all available models
models = list_available_models()
print(f"Available models ({len(models)}):\n")
for model in models:
    print(f"  • {model}")

Available models (12):

  • gpt-oss-20b
  • gpt-oss-120b
  • gpt-4.1-mini
  • gpt-4.1-nano
  • grok-4-fast
  • gemini-2.5-flash-lite
  • llama-3.1-405b-instruct
  • meta-llama-3-1-70b-instruct
  • meta-llama-3-1-8b-instruct
  • deepseek-r1
  • qwen3-14b
  • qwen3-32b


### Functions for the different settings

In [23]:
class ExplanationGeneration_None(dspy.Signature):
    """Extract key properties and attributes from an unfamiliar concept."""
    unfamiliar_concept: str = dspy.InputField(desc="The unfamiliar concept for student to learn")
    familiar_concept: str = dspy.InputField(desc="The familiar concept used to create the analogy")
    Explanation: str = dspy.OutputField(desc="The explanation of how the unfamiliar concept and the familiar concept are analogies according to the properties of each concept")
    #When it is time to test, I would need to collapse the list of explanations into one explanation
class ExplanationGeneration_None_description(dspy.Signature):
    """Extract key properties and attributes from an unfamiliar concept."""
    unfamiliar_concept: str = dspy.InputField(desc="The unfamiliar concept for student to learn")
    description_of_unfamiliar_concept: str = dspy.InputField(desc="The description of the unfamiliar concept")
    familiar_concept: str = dspy.InputField(desc="The familiar concept used to create the analogy")
    description_of_familiar_concept: str = dspy.InputField(desc="The description of the familiar concept")
    Explanation: str = dspy.OutputField(desc="The explanation of how the unfamiliar concept and the familiar concept are analogies according to the properties of each concept")
    #When it is time to test, I would need to collapse the list of explanations into one explanation
class ExplanationGeneration_UnpairedProperties(dspy.Signature):
    """Extract key properties and attributes from an unfamiliar concept."""
    unfamiliar_concept: str = dspy.InputField(desc="The unfamiliar concept for student to learn")
    properties_of_unfamiliar_concept: list[str] = dspy.InputField(desc="List of key properties that characterize the unfamiliar concept. Each property is 1-2 words maximum.")
    familiar_concept: str = dspy.InputField(desc="The familiar concept used to create the analogy")
    properties_of_familiar_concept: list[str] = dspy.InputField(desc="List of key properties that characterize the familiar concept. Each property is 1-2 words maximum.")
    Explanation: list[str] = dspy.OutputField(desc="The explanation of how the unfamiliar concept and the familiar concept are analogies according to each of the properties of each concept. Pair them first and then explain the analogy for each pair.")
class ExplanationGeneration_UnpairedProperties_description(dspy.Signature):
    """Extract key properties and attributes from an unfamiliar concept."""
    unfamiliar_concept: str = dspy.InputField(desc="The unfamiliar concept for student to learn")
    description_of_unfamiliar_concept: str = dspy.InputField(desc="The description of the unfamiliar concept")
    properties_of_unfamiliar_concept: list[str] = dspy.InputField(desc="List of key properties that characterize the unfamiliar concept. Each property is 1-2 words maximum.")
    familiar_concept: str = dspy.InputField(desc="The familiar concept used to create the analogy")
    description_of_familiar_concept: str = dspy.InputField(desc="The description of the familiar concept")
    properties_of_familiar_concept: list[str] = dspy.InputField(desc="List of key properties that characterize the familiar concept. Each property is 1-2 words maximum.")
    Explanation: list[str] = dspy.OutputField(desc="The explanation of how the unfamiliar concept and the familiar concept are analogies according to each of the properties of each concept. Pair them first and then explain the analogy for each pair.")

class ExplanationGeneration_PairedProperties(dspy.Signature):
    """Extract key properties and attributes from an unfamiliar concept."""
    unfamiliar_concept: str = dspy.InputField(desc="The unfamiliar concept for student to learn")
    familiar_concept: str = dspy.InputField(desc="The familiar concept used to create the analogy")
    paired_properties: list[list[str]] = dspy.InputField(desc="Dictionary mapping each unfamiliar concept property to corresponding familiar concept property (1-2 words each)")
    Explanation: list[str] = dspy.OutputField(desc="The explanation of how the unfamiliar concept and the familiar concept are analogies according to each of the paired properties")

class ExplanationGeneration_PairedProperties_description(dspy.Signature):
    """Extract key properties and attributes from an unfamiliar concept."""
    unfamiliar_concept: str = dspy.InputField(desc="The unfamiliar concept for student to learn")
    description_of_unfamiliar_concept: str = dspy.InputField(desc="The description of the unfamiliar concept")
    familiar_concept: str = dspy.InputField(desc="The familiar concept used to create the analogy")
    description_of_familiar_concept: str = dspy.InputField(desc="The description of the familiar concept")
    paired_properties: list[list[str]] = dspy.InputField(desc="Dictionary mapping each unfamiliar concept property to corresponding familiar concept property (1-2 words each)")
    Explanation: list[str] = dspy.OutputField(desc="The explanation of how the unfamiliar concept and the familiar concept are analogies according to each of the paired properties")

#### Examples to Try

In [24]:
adapter = DSPyAdapter(client, model_name="gpt-4.1-nano")
lm = adapter.get_dspy_lm()
dspy.settings.configure(lm=lm)
print("✅ DSPy LM Configured!")

✅ DSPy LM Configured!


In [25]:
unfamiliar_concept = "Respiratory system"
description_of_unfamiliar_concept = "The respiratory system is a complex network of organs and tissues that work together to facilitate the exchange of oxygen and carbon dioxide in the body. It includes the lungs, trachea, bronchi, and bronchioles, among other structures."
familiar_concept = "engine"
description_of_familiar_concept = "An engine is a machine that converts fuel energy into mechanical motion through combustion. It requires fuel, air, and produces exhaust gases."
paired_properties = [["oxygen", "fuel"], ["the lungs", "combustion chamber"], ["breathing muscles", "piston"]]
properties_of_unfamiliar_concept = ["oxygen", "the lungs", "breathing muscles"]
properties_of_familiar_concept = ["fuel", "combustion chamber", "piston"]
Golden_explanation = ['Oxygen corresponds to fuel: In the respiratory system, oxygen is an essential element for sustaining life and participates in the energy-generating process within the organism. In an engine, fuel (such as gasoline, diesel, etc.) is the energy source required for its operation. In this mapping, both oxygen and fuel are critical factors in energy generation within their respective systems.', 'Lungs correspond to the combustion chamber: In the respiratory system, the lungs are responsible for the exchange of oxygen and carbon dioxide, providing oxygen to the organism and expelling waste gases. In an engine, the combustion chamber is where fuel and oxygen mix and burn, releasing energy to drive the engine. In this mapping, the lungs and combustion chamber are the places where gas exchange and energy release occur within their respective systems.', 'Breathing muscles correspond to the piston: In the respiratory system, breathing muscles (such as the diaphragm and intercostal muscles) control the flow of gas into and out of the lungs through contraction and relaxation, enabling the process of breathing. In an engine, the piston converts the energy produced within the combustion chamber into mechanical energy, driving other parts of the engine. In this mapping, both breathing muscles and the piston are crucial components for gas flow and energy conversion within their respective systems.']

In [26]:
## Testing the ExplanationGeneration_None
extractor = dspy.ChainOfThought(ExplanationGeneration_None)
result = extractor(unfamiliar_concept=unfamiliar_concept, familiar_concept=familiar_concept)

#print the results and reasoning
print(f"Explanation: {result.Explanation}")
print(f"Reasoning: {result.reasoning}")


Explanation: The respiratory system and an engine are analogous because both serve as mechanisms that process inputs to produce essential outputs. The respiratory system takes in air, filters and exchanges gases in the lungs, and supplies oxygen to the bloodstream, similar to how an engine takes in fuel and air, combusts them, and produces mechanical power. Both systems are vital for sustaining the function of a larger system— the body in the case of the respiratory system, and a machine or vehicle in the case of an engine.
Reasoning: The respiratory system functions like an engine in that it takes in a vital input—air (oxygen)—and processes it to produce a necessary output—oxygenated blood for the body. Just as an engine intakes fuel and air to generate power, the respiratory system intakes air to facilitate gas exchange, providing the body with the energy it needs to operate.


In [27]:
## Testing the ExplanationGeneration_None_description
extractor = dspy.ChainOfThought(ExplanationGeneration_None_description)
result = extractor(unfamiliar_concept=unfamiliar_concept, description_of_unfamiliar_concept=description_of_unfamiliar_concept, familiar_concept=familiar_concept, description_of_familiar_concept=description_of_familiar_concept)

#print the results and reasoning
print(f"Explanation: {result.Explanation}")
print(f"Reasoning: {result.reasoning}")

Explanation: The respiratory system and an engine are analogous because they both serve as systems that process inputs to produce essential outputs. The respiratory system takes in air (which contains oxygen) through the lungs and exchanges gases to supply oxygen to the body while removing carbon dioxide, similar to how an engine takes in air and fuel to produce mechanical motion. Both systems rely on a network of interconnected parts—lungs, trachea, bronchioles in the respiratory system; cylinders, pistons, and valves in an engine—to facilitate the movement and processing of these inputs. Additionally, both produce byproducts—exhaled gases in the respiratory system and exhaust gases in the engine—highlighting their roles in transforming inputs into useful outputs while managing waste.
Reasoning: The respiratory system functions similarly to an engine because both systems require the intake of a vital resource—oxygen for the respiratory system and fuel and air for the engine—to produce

In [28]:
## Testing the ExplanationGeneration_UnpairedProperties
extractor = dspy.ChainOfThought(ExplanationGeneration_UnpairedProperties)
result = extractor(unfamiliar_concept=unfamiliar_concept, properties_of_unfamiliar_concept=properties_of_unfamiliar_concept, familiar_concept=familiar_concept, properties_of_familiar_concept=properties_of_familiar_concept)

#print the results and reasoning
print(f"Explanation: {result.Explanation}")
print(f"Reasoning: {result.reasoning}")

Explanation: ['Oxygen and fuel: Both the respiratory system and an engine require a vital input—oxygen for the respiratory system and fuel for the engine—to function properly.', 'The lungs and combustion chamber: The lungs facilitate gas exchange, similar to how a combustion chamber ignites fuel, both being central sites for their processes.', 'Breathing muscles and pistons: Breathing muscles help move air in and out of the lungs, comparable to pistons moving within an engine to generate power and facilitate movement.']
Reasoning: The respiratory system can be compared to an engine because both facilitate essential processes for their respective systems. The respiratory system takes in oxygen and expels carbon dioxide, similar to how an engine takes in fuel and air for combustion. The lungs function like a combustion chamber where gas exchange occurs, akin to how an engine's combustion chamber ignites fuel. Breathing muscles support airflow in the respiratory system, comparable to pist

In [29]:
## Testing the ExplanationGeneration_UnpairedProperties_description
extractor = dspy.ChainOfThought(ExplanationGeneration_UnpairedProperties_description)
result = extractor(unfamiliar_concept=unfamiliar_concept, description_of_unfamiliar_concept=description_of_unfamiliar_concept, properties_of_unfamiliar_concept=properties_of_unfamiliar_concept, familiar_concept=familiar_concept, description_of_familiar_concept=description_of_familiar_concept, properties_of_familiar_concept=properties_of_familiar_concept)

#print the results and reasoning
print(f"Explanation: {result.Explanation}")
print(f"Reasoning: {result.reasoning}")

Explanation: ['oxygen - fuel: Both oxygen in the respiratory system and fuel in an engine are essential inputs that enable the primary process to occur.', 'the lungs - combustion chamber: The lungs are the main site for gas exchange, similar to how the combustion chamber is the core site for fuel combustion.', 'breathing muscles - piston: Breathing muscles generate the movement and pressure needed for airflow, akin to how pistons create motion within the engine.']
Reasoning: The respiratory system and an engine both facilitate the exchange and transformation of substances to produce a functional outcome. The lungs in the respiratory system are analogous to the combustion chamber in an engine because both are central sites where a key process occurs—gas exchange in the lungs and combustion in the engine. Breathing muscles are similar to pistons because they generate the necessary movement or pressure to enable airflow or engine motion. The presence of oxygen in the respiratory system is

In [30]:
## Testing the ExplanationGeneration_PairedProperties
extractor = dspy.ChainOfThought(ExplanationGeneration_PairedProperties)
result = extractor(unfamiliar_concept=unfamiliar_concept, familiar_concept=familiar_concept, paired_properties=paired_properties)

#print the results and reasoning
print(f"Explanation: {result.Explanation}")
print(f"Reasoning: {result.reasoning}")

Explanation: ['The lungs are like a combustion chamber because they take in oxygen, similar to how a combustion chamber receives fuel.', 'Breathing muscles are like pistons because they move air in and out, similar to how pistons move within an engine to generate power.', 'Oxygen in the respiratory system is like fuel in an engine because both are essential for producing energy.']
Reasoning: The respiratory system can be compared to an engine because both systems facilitate a vital process—oxygen intake and energy production—through analogous components. The lungs function like a combustion chamber by taking in oxygen, similar to how an engine's combustion chamber receives fuel. Breathing muscles act like pistons, moving air in and out just as pistons move within an engine to generate power. This analogy helps to understand the respiratory system's role in sustaining life by drawing parallels to the familiar operation of an engine.


In [31]:
## Testing the ExplanationGeneration_PairedProperties_description
extractor = dspy.ChainOfThought(ExplanationGeneration_PairedProperties_description)
result = extractor(unfamiliar_concept=unfamiliar_concept, description_of_unfamiliar_concept=description_of_unfamiliar_concept, familiar_concept=familiar_concept, description_of_familiar_concept=description_of_familiar_concept, paired_properties=paired_properties)

#print the results and reasoning
print(f"Explanation: {result.Explanation}")
print(f"Reasoning: {result.reasoning}")

Explanation: ['The lungs are like the combustion chamber because both are sites where a critical exchange occurs—oxygen in the lungs and fuel in the engine.', 'Oxygen is analogous to fuel because both are essential inputs that enable the system to function properly.', 'Breathing muscles are similar to pistons because they move to facilitate the intake and expulsion of air, just as pistons move to generate engine power.']
Reasoning: The respiratory system can be compared to an engine because both involve the intake and processing of essential substances—oxygen for the respiratory system and fuel for the engine—that are necessary for their primary functions. The lungs act like the combustion chamber where oxygen is exchanged, similar to how the combustion chamber in an engine processes fuel and air mixture. Breathing muscles function like pistons, facilitating the movement of air in and out, much like pistons move within an engine to generate motion. This analogy helps to understand the 

### Loading The Dataset

In [32]:
df_scar = pd.read_csv('../../data/SCAR_cleaned_manually.csv')
df_scar.head()


,id,system_a,system_b,system_a_domain,system_b_domain,system_a_background,system_b_background,mappings_parsed,mapping_count,explanation_parsed,explanation_count,system_a_bg_wc,system_b_bg_wc
0,1,biological clock,clock,Biology,Engineering,The biological clock is a fundamental aspect o...,"A clock, also known as a timepiece, is a devic...","[['changes', 'pointer'], ['state', 'time'], ['...",3,['Changes correspond to pointers: In the biolo...,3,190,118
1,2,Biosphere,Library,Biology,Art,The biosphere refers to the sum of all ecosyst...,A library is much more than a building full of...,"[['biology', 'books'], ['biodiversity', 'Book ...",3,"[""Biology corresponds to books: In the biosphe...",3,110,149
2,3,Respiratory system,engine,Biology,Physics,The respiratory system is a critical biologica...,An engine or motor is a machine that transform...,"[['oxygen', 'fuel'], ['the lungs', 'combustion...",3,['Oxygen corresponds to fuel: In the respirato...,3,131,143
3,4,Spread of Pathogens,Spread of Fire,Biology,Physics,"The spread of pathogens is a serious concern, ...",Fire is a natural process that is essential fo...,"[['pathogen', 'fire'], ['crowd', 'combustibles...",3,['Pathogens correspond to fire: in the transmi...,3,127,170
4,5,Gene editing,kirigami,Biology,Art,Gene editing is a revolutionary technique in m...,Kirigami is a fascinating art form that origin...,"[['Gene', 'raw material'], ['CRISPR-Cas9 Techn...",3,['Gene corresponds to raw materials: in gene e...,3,133,130


In [33]:
import ast

# Step 1: Parse the mappings_parsed column (it's stored as string representation of list)
df_scar['mappings_list'] = df_scar['mappings_parsed'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) and x else [])

# Step 2: Extract properties from mappings
# mappings are in format: [['property1_unfamiliar', 'property1_familiar'], ['property2_unfamiliar', 'property2_familiar'], ...]
df_scar['properties_unfamiliar'] = df_scar['mappings_list'].apply(lambda x: [pair[0] for pair in x] if x else [])
df_scar['properties_familiar'] = df_scar['mappings_list'].apply(lambda x: [pair[1] for pair in x] if x else [])

#Step 3: Extract the explanation from the explanation_parsed column
df_scar['explanation_list'] = df_scar['explanation_parsed'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) and x else [])

# Step 4: Clean concept names (remove any domain info in parentheses)
df_scar['unfamiliar_concept'] = df_scar['system_a'].str.strip()
df_scar['familiar_concept'] = df_scar['system_b'].str.strip()

# Step 5: Use the background descriptions as-is
df_scar['description_unfamiliar'] = df_scar['system_a_background']
df_scar['description_familiar'] = df_scar['system_b_background']

print("✅ Data prepared for running functions!")
print(f"\nDataset has {len(df_scar)} rows")
print(f"\nSample row structure:")
print(f"- Unfamiliar Concept: {df_scar.iloc[2]['unfamiliar_concept']}")
print(f"- Familiar Concept: {df_scar.iloc[2]['familiar_concept']}")
print(f"- Properties Unfamiliar: {df_scar.iloc[2]['properties_unfamiliar']}")
print(f"- Properties Familiar: {df_scar.iloc[2]['properties_familiar']}")
print(f"- Description Unfamiliar (first 100 chars): {df_scar.iloc[2]['description_unfamiliar'][:100]}...")
print(f"- Description Familiar (first 100 chars): {df_scar.iloc[2]['description_familiar'][:100]}...")
print(f"- Explanation: {df_scar.iloc[2]['explanation_list']}")

df_scar.head()


✅ Data prepared for running functions!

Dataset has 400 rows

Sample row structure:
- Unfamiliar Concept: Respiratory system
- Familiar Concept: engine
- Properties Unfamiliar: ['oxygen', 'the lungs', 'breathing muscles']
- Properties Familiar: ['fuel', 'combustion chamber', 'piston']
- Description Unfamiliar (first 100 chars): The respiratory system is a critical biological system used for gas exchange in both animals and pla...
- Description Familiar (first 100 chars): An engine or motor is a machine that transforms energy into mechanical energy for various applicatio...
- Explanation: ['Oxygen corresponds to fuel: In the respiratory system, oxygen is an essential element for sustaining life and participates in the energy-generating process within the organism. In an engine, fuel (such as gasoline, diesel, etc.) is the energy source required for its operation. In this mapping, both oxygen and fuel are critical factors in energy generation within their respective systems.', 'Lungs cor

,id,system_a,system_b,system_a_domain,system_b_domain,system_a_background,system_b_background,mappings_parsed,mapping_count,explanation_parsed,...,system_a_bg_wc,system_b_bg_wc,mappings_list,properties_unfamiliar,properties_familiar,explanation_list,unfamiliar_concept,familiar_concept,description_unfamiliar,description_familiar
0,1,biological clock,clock,Biology,Engineering,The biological clock is a fundamental aspect o...,"A clock, also known as a timepiece, is a devic...","[['changes', 'pointer'], ['state', 'time'], ['...",3,['Changes correspond to pointers: In the biolo...,...,190,118,"[[changes, pointer], [state, time], [adjust, m...","[changes, state, adjust]","[pointer, time, maintain]",[Changes correspond to pointers: In the biolog...,biological clock,clock,The biological clock is a fundamental aspect o...,"A clock, also known as a timepiece, is a devic..."
1,2,Biosphere,Library,Biology,Art,The biosphere refers to the sum of all ecosyst...,A library is much more than a building full of...,"[['biology', 'books'], ['biodiversity', 'Book ...",3,"[""Biology corresponds to books: In the biosphe...",...,110,149,"[[biology, books], [biodiversity, Book Type], ...","[biology, biodiversity, ecosystem]","[books, Book Type, library room]",[Biology corresponds to books: In the biospher...,Biosphere,Library,The biosphere refers to the sum of all ecosyst...,A library is much more than a building full of...
2,3,Respiratory system,engine,Biology,Physics,The respiratory system is a critical biologica...,An engine or motor is a machine that transform...,"[['oxygen', 'fuel'], ['the lungs', 'combustion...",3,['Oxygen corresponds to fuel: In the respirato...,...,131,143,"[[oxygen, fuel], [the lungs, combustion chambe...","[oxygen, the lungs, breathing muscles]","[fuel, combustion chamber, piston]",[Oxygen corresponds to fuel: In the respirator...,Respiratory system,engine,The respiratory system is a critical biologica...,An engine or motor is a machine that transform...
3,4,Spread of Pathogens,Spread of Fire,Biology,Physics,"The spread of pathogens is a serious concern, ...",Fire is a natural process that is essential fo...,"[['pathogen', 'fire'], ['crowd', 'combustibles...",3,['Pathogens correspond to fire: in the transmi...,...,127,170,"[[pathogen, fire], [crowd, combustibles], [Pre...","[pathogen, crowd, Prevention and control measu...","[fire, combustibles, Fire fighting methods]",[Pathogens correspond to fire: in the transmis...,Spread of Pathogens,Spread of Fire,"The spread of pathogens is a serious concern, ...",Fire is a natural process that is essential fo...
4,5,Gene editing,kirigami,Biology,Art,Gene editing is a revolutionary technique in m...,Kirigami is a fascinating art form that origin...,"[['Gene', 'raw material'], ['CRISPR-Cas9 Techn...",3,['Gene corresponds to raw materials: in gene e...,...,133,130,"[[Gene, raw material], [CRISPR-Cas9 Technology...","[Gene, CRISPR-Cas9 Technology, edited gene]","[raw material, Scissors, Paper-cut works]",[Gene corresponds to raw materials: in gene ed...,Gene editing,kirigami,Gene editing is a revolutionary technique in m...,Kirigami is a fascinating art form that origin...


### Helper functions to easy execution

In [34]:
"""
Helper functions to run the different explanation generation tasks on df_scar rows
"""
def run_explanation_generation_none(row):
    extractor = dspy.ChainOfThought(ExplanationGeneration_None)
    result = extractor(unfamiliar_concept=row['unfamiliar_concept'], familiar_concept=row['familiar_concept'])
    return result.Explanation, result.reasoning

def run_explanation_generation_none_description(row):
    extractor = dspy.ChainOfThought(ExplanationGeneration_None_description)
    result = extractor(unfamiliar_concept=row['unfamiliar_concept'], description_of_unfamiliar_concept=row['description_unfamiliar'], familiar_concept=row['familiar_concept'], description_of_familiar_concept=row['description_familiar'])
    return result.Explanation, result.reasoning

def run_explanation_generation_unpaired_properties(row):
    extractor = dspy.ChainOfThought(ExplanationGeneration_UnpairedProperties)
    result = extractor(unfamiliar_concept=row['unfamiliar_concept'], properties_of_unfamiliar_concept=row['properties_unfamiliar'], familiar_concept=row['familiar_concept'], properties_of_familiar_concept=row['properties_familiar'])
    return result.Explanation, result.reasoning

def run_explanation_generation_unpaired_properties_description(row):
    extractor = dspy.ChainOfThought(ExplanationGeneration_UnpairedProperties_description)
    result = extractor(unfamiliar_concept=row['unfamiliar_concept'], description_of_unfamiliar_concept=row['description_unfamiliar'], properties_of_unfamiliar_concept=row['properties_unfamiliar'], familiar_concept=row['familiar_concept'], description_of_familiar_concept=row['description_familiar'], properties_of_familiar_concept=row['properties_familiar'])
    return result.Explanation, result.reasoning 

def run_explanation_generation_paired_properties(row):
    extractor = dspy.ChainOfThought(ExplanationGeneration_PairedProperties)
    result = extractor(unfamiliar_concept=row['unfamiliar_concept'], familiar_concept=row['familiar_concept'], paired_properties=row['mappings_list'])
    return result.Explanation, result.reasoning

def run_explanation_generation_paired_properties_description(row):
    extractor = dspy.ChainOfThought(ExplanationGeneration_PairedProperties_description)
    result = extractor(unfamiliar_concept=row['unfamiliar_concept'], description_of_unfamiliar_concept=row['description_unfamiliar'], familiar_concept=row['familiar_concept'], description_of_familiar_concept=row['description_familiar'], paired_properties=row['mappings_list'])
    return result.Explanation, result.reasoning

In [35]:
# Test the helper functions on a row
row = df_scar.iloc[2]
print(f"Unfamiliar Concept: {row['unfamiliar_concept']}")
print(f"Familiar Concept: {row['familiar_concept']}")
print(f"Properties Unfamiliar: {row['properties_unfamiliar']}")
print(f"Properties Familiar: {row['properties_familiar']}")
print(f"Description Unfamiliar: {row['description_unfamiliar']}")
print(f"Description Familiar: {row['description_familiar']}")
print(f"GoldenExplanation: {row['explanation_list']}")

Unfamiliar Concept: Respiratory system
Familiar Concept: engine
Properties Unfamiliar: ['oxygen', 'the lungs', 'breathing muscles']
Properties Familiar: ['fuel', 'combustion chamber', 'piston']
Description Unfamiliar: The respiratory system is a critical biological system used for gas exchange in both animals and plants. In land animals, the lungs play the central role in gas exchange, with millions of air sacs called alveoli serving as the respiratory surface. Oxygen from the environment is brought to the alveoli by breathing muscles, which pump air through a system of hollow tubes, such as the trachea and bronchi, and bronchioles. Birds have similar air sacs called atria, and their bronchioles are termed parabronchi. Fish have gills, while insects have simpler respiratory systems, and amphibians rely on their skin for gas exchange. Plants also have respiratory systems, with stomata serving as one such feature. Understanding how the respiratory system works is critical to comprehendin

In [36]:
print(run_explanation_generation_none(row))
print(run_explanation_generation_none_description(row))
print(run_explanation_generation_unpaired_properties(row))
print(run_explanation_generation_unpaired_properties_description(row))
print(run_explanation_generation_paired_properties(row))
print(run_explanation_generation_paired_properties_description(row))

('The respiratory system and an engine are analogous because both serve as mechanisms that process inputs to produce essential outputs. The respiratory system takes in air, filters and exchanges gases in the lungs, and supplies oxygen to the bloodstream, similar to how an engine takes in fuel and air, combusts them, and produces mechanical power. Both systems are vital for sustaining the function of a larger system— the body in the case of the respiratory system, and a machine or vehicle in the case of an engine.', 'The respiratory system functions like an engine in that it takes in a vital input—air (oxygen)—and processes it to produce a necessary output—oxygenated blood for the body. Just as an engine intakes fuel and air to generate power, the respiratory system intakes air to facilitate gas exchange, providing the body with the energy it needs to operate.')
('The respiratory system and an engine are analogous because they both act as systems that process inputs to produce a vital o

### Evaluation - SBERT

In [37]:
# Import the evaluation module
from explanation_evaluation import ExplanationEvaluator, create_evaluation_summary

# Initialize evaluator
evaluator = ExplanationEvaluator()

# Example: Evaluate a single explanation
def evaluate_single_explanation(golden_explanation, predicted_explanation):
    """Evaluate a single explanation pair."""
    return evaluator.evaluate_explanation(golden_explanation, predicted_explanation)

# Example: Evaluate all explanations in your dataset
def evaluate_all_explanations(df, method_name, predicted_col):
    """Evaluate all explanations for a specific method."""
    results = evaluator.evaluate_batch(
        df, 
        'explanation_list', 
        predicted_col,
        evaluation_type='list_to_list'  # or 'string_to_string' or 'auto'
    )
    return results


✅ Loaded SBERT model: all-mpnet-base-v2


In [38]:
# Test with your sample data
sample_row = df_scar.iloc[2]
golden = ", ".join(sample_row['explanation_list'])
predicted = run_explanation_generation_none(sample_row)[0]

print(type(golden))
print(type(predicted))
result = evaluate_single_explanation(golden, predicted)
print("Sample evaluation result:", result)

<class 'str'>
<class 'str'>
Sample evaluation result: {'sbert_similarity': 0.8435109257698059, 'evaluation_type': 'string_to_string'}


### Running the models - in parallel

In [ ]:
# MODELS = list_available_models()
# setting = "none"
# from tqdm import tqdm
# import json
# import os

# # Create checkpoint directory if it doesn't exist
# os.makedirs('checkpoints/explanation_generation', exist_ok=True)

# TEST_MODE = True  
# subset = df_scar[:5] if TEST_MODE else df_scar

# json_results = []  # ✅ Moved OUTSIDE the loop - accumulates ALL models

# for model in tqdm(MODELS, desc="Models"):
#     print(f"Running {model} with {setting} setting")
    
#     # Configure DSPy with the current model
#     adapter = DSPyAdapter(client, model_name=model)
#     lm = adapter.get_dspy_lm()
#     dspy.settings.configure(lm=lm)
    
#     for idx, row in tqdm(subset.iterrows(), total=len(subset), desc=f"{model} rows"):
#         try:
#             print(f"Running {model} with {setting} setting for row {idx}")
#             result, reasoning = run_explanation_generation_none(row)
#             golden_explanation = ", ".join(row['explanation_list'])
#             evaluation_result = evaluate_single_explanation(golden_explanation, result)
#             json_results.append({
#                 "model": model,
#                 "setting": setting,
#                 "row_index": idx,
#                 "error": None,
#                 "unfamiliar_concept": row['unfamiliar_concept'],    
#                 "familiar_concept": row['familiar_concept'],
#                 "description_unfamiliar": row['description_unfamiliar'],
#                 "description_familiar": row['description_familiar'],
#                 "properties_unfamiliar": row['properties_unfamiliar'],
#                 "properties_familiar": row['properties_familiar'],
#                 "explanation_list": row['explanation_list'],
#                 "result": result,
#                 "reasoning": reasoning,
#                 "evaluation_result": evaluation_result
#             })
#         except Exception as e:
#             print(f"❌ Error processing row {idx} with {model}: {str(e)}")
#             json_results.append({
#                 "model": model,
#                 "setting": setting,
#                 "row_index": idx,
#                 "error": str(e),
#                 "unfamiliar_concept": row['unfamiliar_concept'],    
#                 "familiar_concept": row['familiar_concept'],
#                 "description_unfamiliar": row['description_unfamiliar'],
#                 "description_familiar": row['description_familiar'],
#                 "properties_unfamiliar": row['properties_unfamiliar'],
#                 "properties_familiar": row['properties_familiar'],
#                 "explanation_list": row['explanation_list'],
#                 "result": "",
#                 "reasoning": "",
#                 "evaluation_result": ""
#             })  
#             continue 
            
#     # ✅ Still save individual model checkpoints
#     model_results = [r for r in json_results if r['model'] == model]
#     if model_results:
#         checkpoint_df = pd.DataFrame(model_results)
#         checkpoint_df.to_csv(f'checkpoints/explanation_generation/{setting}_{model}_checkpoint.csv', index=False)
#         with open(f'checkpoints/explanation_generation/{setting}_{model}_checkpoint.json', 'w') as f:
#             json.dump(model_results, f, indent=2)

# # ✅ After ALL models complete - save combined results
# print(f"\n✅ All models completed! Total results: {len(json_results)}")
# combined_df = pd.DataFrame(json_results)
# combined_df.to_csv(f'checkpoints/explanation_generation/{setting}_ALL_MODELS_combined.csv', index=False)
# with open(f'checkpoints/explanation_generation/{setting}_ALL_MODELS_combined.json', 'w') as f:
#     json.dump(json_results, f, indent=2)
# print(f"✅ Saved combined results to {setting}_ALL_MODELS_combined.csv/json")

# # ✅ Optional: Print summary statistics
# print("\n📊 Summary by Model:")
# summary = combined_df.groupby('model').agg({
#     'row_index': 'count',
#     'error': lambda x: x.notna().sum()
# }).rename(columns={'row_index': 'total_rows', 'error': 'errors'})
# print(summary)

In [ ]:
MODELS = list_available_models()
setting = "none"
from tqdm import tqdm
import json
import os  # ✅ Add this import

# ✅ Create checkpoint directory if it doesn't exist
os.makedirs('checkpoints/explanation_generation', exist_ok=True)

TEST_MODE = True  
subset = df_scar[:5] if TEST_MODE else df_scar

for model in tqdm(MODELS, desc="Models"):
    json_results = []
    print(f"Running {model} with {setting} setting")
    
    # Configure DSPy with the current model
    adapter = DSPyAdapter(client, model_name=model)
    lm = adapter.get_dspy_lm()
    dspy.settings.configure(lm=lm)
    
    for idx, row in tqdm(subset.iterrows(), total=len(subset), desc=f"{model} rows"):
        try:
            print(f"Running {model} with {setting} setting for row {idx}")
            result, reasoning = run_explanation_generation_none(row)
            golden_explanation = ", ".join(row['explanation_list'])
            evaluation_result = evaluate_single_explanation(golden_explanation, result)
            json_results.append({
                "model": model,
                "setting": setting,
                "row_index": idx,
                "error": None,
                "unfamiliar_concept": row['unfamiliar_concept'],    
                "familiar_concept": row['familiar_concept'],
                "description_unfamiliar": row['description_unfamiliar'],
                "description_familiar": row['description_familiar'],
                "properties_unfamiliar": row['properties_unfamiliar'],
                "properties_familiar": row['properties_familiar'],
                "explanation_list": row['explanation_list'],
                "result": result,
                "reasoning": reasoning,
                "sbert_score": evaluation_result.sbert_score,
                "Type_of_sbert_score": evaluation_result.type_of_sbert_score
            })
        except Exception as e:
            print(f"❌ Error processing row {idx} with {model}: {str(e)}")
            json_results.append({
                "model": model,
                "setting": setting,
                "row_index": idx,
                "error": str(e),
                "unfamiliar_concept": row['unfamiliar_concept'],    
                "familiar_concept": row['familiar_concept'],
                "description_unfamiliar": row['description_unfamiliar'],
                "description_familiar": row['description_familiar'],
                "properties_unfamiliar": row['properties_unfamiliar'],
                "properties_familiar": row['properties_familiar'],
                "explanation_list": row['explanation_list'],
                "result": "",
                "sbert_score": "",
                "Type_of_sbert_score": ""
            })  
            continue 
            
    # After each model completes
    if json_results:
        checkpoint_df = pd.DataFrame(json_results)
        checkpoint_df.to_csv(f'checkpoints/explanation_generation/{setting}_{model}_checkpoint.csv', index=False)
        with open(f'checkpoints/explanation_generation/{setting}_{model}_checkpoint.json', 'w') as f:
            json.dump(json_results, f, indent=2)

Models:   0%|          | 0/12 [00:00<?, ?it/s]

Running gpt-oss-20b with none setting


Running gpt-oss-20b with none setting for row 0


Running gpt-oss-20b with none setting for row 1


Running gpt-oss-20b with none setting for row 2


...

Running qwen3-32b with none setting for row 1


Running qwen3-32b with none setting for row 2


Running qwen3-32b with none setting for row 3


Running qwen3-32b with none setting for row 4


Models: 100%|██████████| 12/12 [09:13<00:00, 46.13s/it]


In [42]:
import pandas as pd
import glob

# Read all checkpoint CSV files
checkpoint_files = glob.glob('checkpoints/explanation_generation/none_*_checkpoint.csv')
print(f"Found {len(checkpoint_files)} checkpoint files:")
for f in checkpoint_files:
    print(f"  • {f}")

# Combine all DataFrames
all_dfs = []
for file in checkpoint_files:
    df = pd.read_csv(file)
    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)
print(f"\n✅ Combined {len(combined_df)} total results from {combined_df['model'].nunique()} models")

# Save combined results
combined_df.to_csv('checkpoints/explanation_generation/none_ALL_MODELS_combined.csv', index=False)
print("✅ Saved combined results!")

# Show summary
print("\n📊 Summary:")
print(combined_df.groupby('model')['row_index'].count())

Found 12 checkpoint files:
  • checkpoints/explanation_generation\none_deepseek-r1_checkpoint.csv
  • checkpoints/explanation_generation\none_gemini-2.5-flash-lite_checkpoint.csv
  • checkpoints/explanation_generation\none_gpt-4.1-mini_checkpoint.csv
  • checkpoints/explanation_generation\none_gpt-4.1-nano_checkpoint.csv
  • checkpoints/explanation_generation\none_gpt-oss-120b_checkpoint.csv
  • checkpoints/explanation_generation\none_gpt-oss-20b_checkpoint.csv
  • checkpoints/explanation_generation\none_grok-4-fast_checkpoint.csv
  • checkpoints/explanation_generation\none_llama-3.1-405b-instruct_checkpoint.csv
  • checkpoints/explanation_generation\none_meta-llama-3-1-70b-instruct_checkpoint.csv
  • checkpoints/explanation_generation\none_meta-llama-3-1-8b-instruct_checkpoint.csv
  • checkpoints/explanation_generation\none_qwen3-14b_checkpoint.csv
  • checkpoints/explanation_generation\none_qwen3-32b_checkpoint.csv

✅ Combined 60 total results from 12 models
✅ Saved combined results!